# 实验六 · 模型推理 —— 从 ATC 到 aclmdl

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：50~60 分钟

本实验要接进应用的是一整张计算图：几十上百个算子、一份权重、一套执行顺序。一个训练好的模型不能直接在昇腾上执行。它要先经过 **GE（Graph Engine，图引擎）** 编译，做图优化、算子调度优化与权重数据重排，产出一个 `.om` 离线模型；之后应用侧用 `aclmdl` 一族接口加载并执行它。

本实验用的是 **ResNet-18**：一个在 ImageNet 上训练过的图像分类网络，输入是 224×224 的三通道图像，输出是 1000 个类别的得分。测试数据是两张真实的照片。真实模型**推理的结果有语义，可以直接看出对错**。

> **实验说明**
> 1. 本实验的核心内容有五点：ATC 转换、四种加载方式的取舍、Dataset 与 DataBuffer 的组织、释放顺序，以及动态 Batch。
> 2. **本实验需要联网下载模型与图片**（约 45 MB），并需要 Python 的 `Pillow`、`numpy` 与 `onnxruntime`。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。§8 的两次 ATC 转换各需要一至两分钟。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**，并需要 `atc` 命令可用。
> 5. 本实验的主机程序**按模型描述编写**，不针对某一个模型：固定 Batch 与动态 Batch 两个模型由同一个二进制处理。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说出一个训练好的模型要经过哪一步才能在昇腾上执行，以及那一步做了哪三类优化
- 写出 ATC 的基本命令，并说明 `--framework`、`--soc_version` 与 `--input_format` 各自的含义
- 把一张 JPEG 图片处理成模型要的输入：等比缩放、中心裁剪、归一化，以及从 HWC 到 NCHW 的转置
- 说明预处理的参数为什么由模型的训练配置决定，而不能自行选定
- 在四种加载接口之间按**模型数据来源**与**运行内存的管理者**两个维度做出选择
- 说明 `aclmdlQuerySize` 查不出大小的是哪一类模型，以及用 `--dynamic_batch_size` 构建的模型为什么不属于这一类
- 用 `aclmdlDesc` 查出模型的输入输出个数、名字、字节数与维度，并据此分配内存
- 按名字而不是按顺序对齐多输入多输出
- 按规定的次序释放资源：先取地址、再释放内存、再销毁 DataBuffer、最后销毁 Dataset；先 Unload 再释放运行内存
- 从模型的 1000 个得分里取出 Top-5，并说明输出为什么是 `[batch, 1000]` 这个形状
- 设置动态 Batch，并解释为什么增大 Batch 会同时提高吞吐与单次执行的时延


## 🗺️ 学习路径

1. **准备阶段**：理解框架模型为什么不能直接执行，以及编译这一步做了哪三类只能在编译期做的事
2. **接口全貌**：四个加载接口的两个划分维度，以及限制这一选择的那条约束究竟约束了什么
3. **数据组织**：模型规格、数据集合与单个张量三者的关系，按名字对齐，以及释放为什么有固定次序
4. **素材准备**：把一张 JPEG 变成模型要的字节序列，理解预处理的每个参数都由模型的训练配置决定
5. **程序实现**：写一个按模型描述工作的程序，使固定 Batch 与动态 Batch 两个模型共用同一个二进制
6. **测量与判读**：把一次推理拆成量级不同的几段分别报出，并由分类结果判断整条通路是否正确
7. **动态 Batch**：理解批大小增大时吞吐与单次执行时延为什么同时上升，以及取舍体现在哪里


## 1. 模型从哪里来

一个用 PyTorch、TensorFlow 或 MindSpore 训练出来的模型，描述的是**一张计算图**：哪些算子、按什么顺序、参数是什么。它与具体的硬件无关，因此也不能直接在昇腾上执行。

中间这一步由 **GE（Graph Engine，图引擎）** 完成。它把开源框架的模型编译成一个适配昇腾 AI 处理器的**离线模型**（`.om` 文件），过程中做三类优化：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 优化 | 做的事 | 为什么必须在编译期做 |
| --- | --- | --- |
| 图优化 | 算子融合、常量折叠、无用节点消除 | 融合要改变图的结构，运行期改不动 |
| 算子调度优化 | 决定算子在哪些核上、按什么顺序执行 | 调度依赖于对整张图的全局视野 |
| 权重数据重排 | 把权重按硬件偏好的格式重新摆放 | 重排一次可以为每一次推理省下数据搬运 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">优化</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做的事</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">为什么必须在编译期做</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图优化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子融合、常量折叠、无用节点消除</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">融合要改变图的结构，运行期改不动</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">算子调度优化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">决定算子在哪些核上、按什么顺序执行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调度依赖于对整张图的全局视野</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">权重数据重排</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把权重按硬件偏好的格式重新摆放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">重排一次可以为每一次推理省下数据搬运</td>
</tr>
</tbody>
</table>

第三类优化说明了离线模型这一名称的含义：权重在 `.om` 里已经是硬件需要的排布，加载之后可以直接使用；若放在运行期重排，每加载一次模型都要重做一遍。

编译有两条路径：**ATC 命令行**（本章采用）与图开发接口（详见官方图开发文档）。

编译出 `.om` 之后，应用侧的调用次序是固定的：

<img src="images/07.06_infer_overview.png" alt="模型推理应用的接口调用流程" width="400">

流程图勾出了整个实验的步骤：**初始化与去初始化在最外层，模型加载与卸载在中间，模型执行在最内层**。图中标注的「如果存在多张图片，则循环执行此框中的流程」正是本实验处理两张照片的方式：模型只加载一次，执行两次。绿色的两个可选步骤本实验不涉及，媒体数据处理由 §7.2 在主机侧用 Python 完成。

### 1.1 本实验用的模型与图片

**ResNet-18** 是一个 18 层的残差卷积网络，在 ImageNet 上训练，把一张 224×224 的三通道图像分到 1000 个类别中去。它的规模足以让本实验测到的每一个数字都有现实意义：权重数十 MB，一次推理是毫秒量级，而不是微秒。

测试数据是两张真实的照片。**一个训练好的分类模型对同一张图片给出什么类别是确定的**，因此本实验可以在主机侧用同一个 ONNX 模型算出一份参考结果，再拿它去核对 NPU 的输出（§7.3）。**这样得到的判据不依赖于任何写在正文里的类别标识**：换一张图片、换一个模型，判据都仍然成立。

### 1.2 本实验的范围

本实验要回答三个问题：**一次推理的时间分布在哪些环节**、**运行内存由谁管理**、**增大 Batch 的收益与代价各是什么**。§16 逐条给出实测的回答。

不在范围内的有三项：不评估模型的分类准确率，那是训练的问题；预处理全部在主机侧用 Python 完成，不涉及设备侧的图像处理接口；只使用同步的 `aclmdlExecute`，不做异步与多流。


## 2. 模型编译：ATC

```bash
atc --model=resnet18_Opset16.onnx --framework=5 --output=resnet18 \
    --soc_version=<设备型号> --input_format=NCHW --output_type=FP32
```

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 参数 | 含义 | 本实验取值 |
| --- | --- | --- |
| `--model` | 输入的框架模型文件 | `resnet18_Opset16.onnx` |
| `--framework` | 框架类型：0=Caffe，1=MindSpore，3=TensorFlow，**5=ONNX** | 5 |
| `--output` | 输出的 `.om` 文件名（不带扩展名） | `resnet18` |
| `--soc_version` | 昇腾 AI 处理器版本，**必须与运行设备一致** | 由 §6 从设备读出 |
| `--input_format` | 输入数据的排布格式 | `NCHW`（图像模型） |
| `--output_type` | 输出数据类型 | `FP32` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验取值</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--model</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入的框架模型文件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>resnet18_Opset16.onnx</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--framework</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">框架类型：0=Caffe，1=MindSpore，3=TensorFlow，<strong>5=ONNX</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">5</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--output</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出的 <code>.om</code> 文件名（不带扩展名）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>resnet18</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--soc_version</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">昇腾 AI 处理器版本，<strong>必须与运行设备一致</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由 §6 从设备读出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--input_format</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入数据的排布格式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>NCHW</code>（图像模型）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--output_type</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输出数据类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>FP32</code></td>
</tr>
</tbody>
</table>

动态 Batch 再加两个参数，§5 会展开：

```bash
atc ... --input_shape="x:-1,3,224,224" --dynamic_batch_size="1,2,4,8"
```

`--input_format` 决定了模型期望输入按什么顺序排列：`NCHW` 表示批、通道、高、宽。**这个参数与 §7.2 的转置是同一件事的两端**：转换时声明了 NCHW，送入数据时就必须按 NCHW 排布。

`--soc_version` 体现的是**编译产物与产品型号绑定**：这个参数填错时 ATC 仍会成功，模型加载才会失败。


## 3. 四种加载方式

两个维度，四个接口：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 接口 | 模型数据来源 | 运行内存的管理者 |
| --- | --- | --- |
| `aclmdlLoadFromFile` | 文件 | 系统管理 |
| `aclmdlLoadFromMem` | 内存 | 系统管理 |
| `aclmdlLoadFromFileWithMem` | 文件 | **调用者管理**（工作内存 + 权值内存） |
| `aclmdlLoadFromMemWithMem` | 内存 | **调用者管理** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">模型数据来源</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">运行内存的管理者</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlLoadFromFile</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">文件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">系统管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlLoadFromMem</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">系统管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlLoadFromFileWithMem</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">文件</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>调用者管理</strong>（工作内存 + 权值内存）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlLoadFromMemWithMem</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>调用者管理</strong></td>
</tr>
</tbody>
</table>

**运行内存**分两块，用途不同：

- **工作内存**：模型执行过程中的临时数据，每执行一次都要用；
- **权值内存**：存放权重，加载后就不变。

这两块内存的实际大小会在 §14 打印出来。**权值内存约等于模型参数量乘以每个参数的字节数**，工作内存则由中间特征图的峰值决定。

### 3.1 限制这一选择的一条约束

由调用者管理内存时，必须先调用 `aclmdlQuerySize` 查出这两块内存的大小。这个接口有一条限制：

> **如果模型输入数据的 Shape 不确定，则不能调用 `aclmdlQuerySize` 查询内存大小。**

「Shape 不确定」指的是**编译期无法确定内存上界**的那一类，主要有两种：用 Shape 范围（`--input_shape_range`）构建的模型，以及不生成 `.om`、直接把构图结果放在内存里的网络。

**用 `--dynamic_batch_size` 构建的动态 Batch 模型不属于这一类。** 可选的批大小在构建时就已经逐个列举，最大批大小需要多少内存可以算出，因此它仍然可以查、也仍然可以由调用者管理内存。§14 把两个模型各查一次，把这条界线实测出来。

### 3.2 注意参数顺序

```cpp
// 查询：出参顺序是（工作内存, 权值内存）
aclmdlQuerySize(path, &workSize, &weightSize);

// 加载：入参顺序是（工作内存指针, 工作内存大小, 权值内存指针, 权值内存大小）
aclmdlLoadFromFileWithMem(path, &modelId, workPtr, workSize, weightPtr, weightSize);
```

两者的次序是一致的，都是先工作内存后权值内存，**但两块内存的大小通常相差很多**。若次序颠倒，权值内存不足以存放权重，工作内存又超出实际需要。

## 4. 数据组织与释放顺序

### 4.1 三个数据类型

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 类型 | 表示什么 | 关键接口 |
| --- | --- | --- |
| `aclmdlDesc` | **模型规格**：几个输入、几个输出、各自的名字、字节数、维度 | `aclmdlCreateDesc` / `aclmdlGetDesc` |
| `aclmdlDataset` | 一次执行的**输入集合**或**输出集合** | `aclmdlCreateDataset` / `aclmdlAddDatasetBuffer` |
| `aclDataBuffer` | **单个张量**的设备地址与字节数 | `aclCreateDataBuffer` / `aclGetDataBufferAddr` / `aclGetDataBufferSizeV2` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">表示什么</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">关键接口</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlDesc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>模型规格</strong>：几个输入、几个输出、各自的名字、字节数、维度</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlCreateDesc</code> / <code>aclmdlGetDesc</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlDataset</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">一次执行的<strong>输入集合</strong>或<strong>输出集合</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlCreateDataset</code> / <code>aclmdlAddDatasetBuffer</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclDataBuffer</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>单个张量</strong>的设备地址与字节数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclCreateDataBuffer</code> / <code>aclGetDataBufferAddr</code> / <code>aclGetDataBufferSizeV2</code></td>
</tr>
</tbody>
</table>

三者的包含关系是：

<img src="images/07.06_dataset.png" alt="aclmdlDataset 与 aclDataBuffer 的关系" width="400">

这张图说明了两件事：**一个 `aclmdlDataset` 里挂多个 `aclDataBuffer`**，顺序与模型的输入（或输出）序号一一对应；而**一个 `aclDataBuffer` 只有地址与长度两个字段**，它不持有数据类型、形状这些信息，那些要向 `aclmdlDesc` 查询。

**本实验的 `CreateDataset` 完全按模型描述工作**：输入几个、每个多大，都从 `aclmdlDesc` 里问出来。固定 Batch 的模型有一个输入，动态 Batch 的模型有两个，同一段代码都能建对。

> 取长度要用 **`aclGetDataBufferSizeV2`**。`aclGetDataBufferSize` 在后续版本会废弃。

### 4.2 多输入多输出要按名字对齐

模型有多个输入时，**不要假定顺序**。正确的做法是先用 `aclmdlGetInputNameByIndex` / `aclmdlGetOutputNameByIndex` 拿到名称，再按名称找到对应的 index：

```cpp
size_t index = 0;
aclmdlGetInputIndexByName(modelDesc, "x", &index);
```

动态 Batch 的批大小选择器就是靠这个机制找到的，名称是固定的 `ACL_DYNAMIC_TENSOR_NAME`（§5）。

### 4.3 释放顺序：两条固定的次序

**输出侧：**

$$\text{aclGetDataBufferAddr 取地址} \to \text{aclrtFree(地址)} \to \text{aclDestroyDataBuffer} \to \text{aclmdlDestroyDataset}$$

**次序不可颠倒。** 设备内存的地址只保存在 DataBuffer 中，先销毁 DataBuffer 之后就无法再取得该地址，那块内存也就无法释放。

**卸载侧：**

$$\text{aclmdlUnload} \to \text{aclmdlDestroyDesc} \to \text{释放工作内存} \to \text{释放权值内存}$$

**必须先 Unload 再释放它的运行内存。** 若次序颠倒，运行时仍然持有那两块已经释放的内存。

两条次序遵循同一条原则：**销毁一个持有引用的对象之前，先处理完它所引用的资源。**


## 5. 动态 Batch

### 5.1 可选的批大小不是任意值

ATC 转换时用 `--dynamic_batch_size="1,2,4,8"` 声明四个可选的**批大小**（Batch Size）。运行期只能从这四个值里选一个，不能填 3、也不能填 16。ATC 的参数说明把这几个可选值称为「档位」，本实验统一写作批大小。

只能取这几个值的原因是 GE 要为每一个批大小编译一份执行方案：可选的批大小越多，`.om` 越大、编译耗时越长。**这是编译期的开销与运行期灵活性之间的一次取舍。**

模型支持哪些批大小可以查询出来：

```cpp
aclmdlBatch gears = {};
aclmdlGetDynamicBatch(modelDesc, &gears);   // gears.batchCount, gears.batch[i]
```

### 5.2 批大小选择器是一个额外的输入

转成动态 Batch 之后，模型会**多出一个输入**，专门用来承载批大小的选择。它的名字是固定的 `ACL_DYNAMIC_TENSOR_NAME`：

```cpp
size_t index = 0;
aclmdlGetInputIndexByName(modelDesc, ACL_DYNAMIC_TENSOR_NAME, &index);
aclmdlSetDynamicBatchSize(modelId, input, index, batchSize);
```

**两条规则：**

1. 设置的 batch size 只能是构建时声明的可选值之一。
2. **批大小选择器对应的那块内存在申请之后不要写入任何数据**，其内容由系统填写；写入数据会导致批大小的设置不生效。本实验的 `CreateDataset` 因此对这一个缓冲区跳过了清零。

§11 的 `info` 模式会把这个额外输入打印出来：**动态 Batch 模型的输入个数比固定 Batch 的版本多一个**。

### 5.3 输出缓冲区按最大的批大小分配

Dataset 只建一次，而不同批大小的输出大小不同。因此 `aclmdlGetOutputSizeByIndex` 在动态模型上报的是**最大批大小所需的字节数**，缓冲区按它分配，较小的批大小只用其中一部分。

执行之后可以用 `aclmdlGetCurOutputDims` 查出这一次实际的输出维度：**缓冲区的容量与本次实际使用的字节数是两个不同的量**。§15 把这两个数并排打印。


## 6. 环境准备与检查

先建目录、导入 CANN 环境变量。本实验要建三个目录：源码、模型、数据。


In [ ]:
!mkdir -p src_model model data

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


检查 `atc` 与 `g++`，并从设备读出型号：**ATC 的 `--soc_version` 必须与它一致**（§2）。


In [ ]:
import os, shutil, subprocess, sys

print("atc      :", shutil.which("atc") or "⚠️  未找到")
print("g++      :", shutil.which("g++") or "⚠️  未找到")

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]
ACL_LIBDIRS = ["-L" + path for path in lib_dirs]


def find_lib(candidates):
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


selected = []
for purpose, candidates in [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("模型管理", ["acl_mdl", "ascendcl"]),
]:
    name = find_lib(candidates)
    print(f"{purpose:<10} 候选 {candidates} -> {name}")
    if name is not None and name not in selected:
        selected.append(name)
ACL_LIBS = ["-l" + name for name in selected]
print("链接参数  :", " ".join(ACL_LIBS))

for extra in os.environ.get("PYTHONPATH", "").split(":"):
    if extra and extra not in sys.path:
        sys.path.append(extra)
SOC_VERSION = "Ascend910B1"  # ← 读不出来时的回退值，请按本机型号修改
try:
    import acl

    acl.init()
    SOC_VERSION = acl.get_soc_name()
    acl.finalize()
except Exception as exc:
    print("⚠️  pyACL 读取型号失败，使用回退值：", exc)
print("设备型号  :", SOC_VERSION)


预处理要用到 `Pillow` 与 `numpy`，缺失时先安装。主机侧参考推理还要用到 `onnxruntime`，在 §7.3 安装。


In [ ]:
import importlib, subprocess, sys

for module, package in [("PIL", "Pillow"), ("numpy", "numpy")]:
    try:
        importlib.import_module(module)
        print(f"{module:<8} 已就绪")
    except ImportError:
        print(f"{module:<8} 缺失，开始安装 {package}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package], check=False
        )


## 7. 准备模型与图片

### 7.1 素材：联网下载或离线放置

本实验需要四个文件：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 文件 | 用途 | 来源 | 大小 |
| --- | --- | --- | --- |
| <code>model/resnet18_Opset16.onnx</code> | 待转换的模型；同时用于 §7.3 的主机侧参考 | ONNX Model Zoo（Git LFS） | 约 45 MB |
| <code>data/dog1_1024_683.jpg</code> | 测试图片 | 昇腾样例素材 | 约 35 KB |
| <code>data/dog2_1024_683.jpg</code> | 测试图片 | 昇腾样例素材 | 约 40 KB |
| <code>data/imagenet_classes.txt</code> | 1000 个类别的名称，仅用于显示 | PyTorch Hub | 约 10 KB |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">文件</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">用途</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">来源</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">大小</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>model/resnet18_Opset16.onnx</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">待转换的模型；同时用于 §7.3 的主机侧参考</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ONNX Model Zoo（Git LFS）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">约 45 MB</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>data/dog1_1024_683.jpg</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">测试图片</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">昇腾样例素材</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">约 35 KB</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>data/dog2_1024_683.jpg</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">测试图片</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">昇腾样例素材</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">约 40 KB</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>data/imagenet_classes.txt</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1000 个类别的名称，仅用于显示</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">PyTorch Hub</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">约 10 KB</td>
</tr>
</tbody>
</table>

下面的单元格按**先看本地、再考虑下载**的次序工作，因此**两种准备方式都可以**：

- **联网**：直接运行，缺哪个下载哪个，已经就位的不会重复拉取；
- **离线**：把四个文件预先放到上表的路径下再运行；也可以把它们放在同一个目录里，
  把该目录填进单元格里的 `OFFLINE_DIR`，由单元格复制到位。

模型文件在仓库中由 **Git LFS** 存放，网页地址打开的是页面而不是文件，
因此下载用的是 `raw` 形式的地址，并且必须跟随重定向。
**取到的文件若明显偏小，多半是 LFS 指针文件或一个错误页面**，
单元格因此对每个文件核对大小的下限，并在不达标时删除该文件，
使下一次运行不会把上一次的残留当成已就绪。

若实验机无法访问外网，单元格会列出还缺哪些文件、各自的下载地址与目标路径，
按提示放好之后重新运行本单元格即可。


In [ ]:
import os, shutil, subprocess, urllib.request

ONNX_PATH = "model/resnet18_Opset16.onnx"
LABEL_PATH = "data/imagenet_classes.txt"

# 离线准备时，把四个文件放进同一个目录，再把该目录填在这里（按文件名匹配）。
# 留空表示不使用这条路径。
OFFLINE_DIR = ""

# 第四列是文件大小的下限（KB）：小于它说明取到的不是目标文件
ASSETS = [
    (
        ONNX_PATH,
        "https://github.com/onnx/models/raw/refs/heads/main/Computer_Vision/"
        "resnet18_Opset16_timm/resnet18_Opset16.onnx",
        40000,
    ),
    (
        "data/dog1_1024_683.jpg",
        "https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/"
        "dog1_1024_683.jpg",
        10,
    ),
    (
        "data/dog2_1024_683.jpg",
        "https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/"
        "dog2_1024_683.jpg",
        10,
    ),
    (
        LABEL_PATH,
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        5,
    ),
]


def big_enough(path, min_kb):
    "文件存在且不小于下限时返回它的大小（KB），否则返回 None。"
    if not os.path.exists(path):
        return None
    size_kb = os.path.getsize(path) / 1024
    return size_kb if size_kb >= min_kb else None


def download(url, path):
    "下载一个文件。优先用 wget，没有 wget 时退回 urllib。失败时删除残留。"
    if shutil.which("wget"):
        proc = subprocess.run(
            ["wget", "-q", "-O", path, url], capture_output=True, text=True
        )
        if proc.returncode == 0:
            return ""
        reason = proc.stderr.strip().splitlines()[-1:] or ["wget 返回码 %d" % proc.returncode]
        note = reason[0]
    else:
        try:
            urllib.request.urlretrieve(url, path)
            return ""
        except Exception as exc:  # noqa: BLE001 —— 网络故障的形态很多，一并记下
            note = "%s: %s" % (type(exc).__name__, exc)
    if os.path.exists(path):
        os.remove(path)
    return note


missing = []
for path, url, min_kb in ASSETS:
    os.makedirs(os.path.dirname(path), exist_ok=True)

    # ① 本地已就位
    size_kb = big_enough(path, min_kb)

    # ② 从离线目录复制
    from_offline = False
    if size_kb is None and OFFLINE_DIR:
        source = os.path.join(OFFLINE_DIR, os.path.basename(path))
        if os.path.exists(source):
            shutil.copyfile(source, path)
            from_offline = True
            size_kb = big_enough(path, min_kb)
            if size_kb is None:
                os.remove(path)

    # ③ 联网下载。离线目录里已经有这个文件、只是不合格时不再联网，直接报出原因
    if size_kb is None:
        if from_offline:
            note = "离线目录里的同名文件只有 %.1f KB，小于下限" % (
                os.path.getsize(os.path.join(OFFLINE_DIR, os.path.basename(path))) / 1024
            )
        else:
            if os.path.exists(path):  # 上一次留下的残缺文件，先清掉
                os.remove(path)
            note = download(url, path)
            size_kb = big_enough(path, min_kb)
        if size_kb is None:
            if os.path.exists(path):
                os.remove(path)
            missing.append((path, url, min_kb, note or "取到的文件小于大小下限"))
            print("❌ %-30s %s" % (path, missing[-1][3][:80]))
            continue

    print("✅ %-30s %9.1f KB" % (path, size_kb))

ASSETS_OK = not missing
if missing:
    print("\n以下文件需要离线准备。请分别下载后放到对应路径，或放进同一个目录并填写 OFFLINE_DIR：")
    for path, url, min_kb, _ in missing:
        print("  路径 %s   （不小于 %d KB）\n  地址 %s\n" % (path, min_kb, url))
    print("放好之后重新运行本单元格。")


### 7.2 预处理：从 JPEG 到 NCHW

模型要的不是 JPEG，而是一段 `1×3×224×224` 的 float32 数据。中间有五步：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | 做的事 | 为什么 |
| --- | --- | --- |
| 等比缩放 | 把短边缩到 256，长边按同一比例缩放 | 保持原图的长宽比；直接缩成正方形会使画面变形 |
| 中心裁剪 | 从缩放后的图正中取 224×224 | 模型的输入尺寸是 224×224 |
| 归一化 | 每个通道除以 255 | 把 0–255 的整数像素值映射到 0–1 |
| 标准化 | 每个通道减去均值、再除以标准差 | 使输入的分布与模型训练时一致；这两组常数由模型的训练配置决定 |
| 转置 | 从 HWC 排布换成 NCHW | 图片解码后的排布是高、宽、通道，模型要求的是批、通道、高、宽 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">步骤</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做的事</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">为什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">等比缩放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把短边缩到 256，长边按同一比例缩放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">保持原图的长宽比；直接缩成正方形会使画面变形</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">中心裁剪</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">从缩放后的图正中取 224×224</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型的输入尺寸是 224×224</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">归一化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个通道除以 255</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把 0–255 的整数像素值映射到 0–1</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">标准化</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个通道减去均值、再除以标准差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">使输入的分布与模型训练时一致；这两组常数由模型的训练配置决定</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转置</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">从 HWC 排布换成 NCHW</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图片解码后的排布是高、宽、通道，模型要求的是批、通道、高、宽</td>
</tr>
</tbody>
</table>

标准化用的均值 `(0.485, 0.456, 0.406)` 与标准差 `(0.229, 0.224, 0.225)` 是 ImageNet 上训练的视觉模型普遍采用的一组常数。**它们属于模型的训练配置，不是可以自行选定的参数**：省掉这一步或换一组数，程序不会报错，字节数也不变，模型读到的却是一张分布被移动过的图像，分类结果因此可能偏移到相近的类别上。换一个模型时，应当查阅该模型自己的说明。

转置做错的后果更彻底：同样的 602112 个字节，排列顺序不同，模型读到的就是完全不同的图像。§2 的 `--input_format=NCHW` 声明的正是这个顺序。

处理后的数据写成 `.bin`，一张图 $3 \times 224 \times 224 \times 4 = 602112$ 字节。


In [ ]:
import numpy as np
from PIL import Image

CHANNELS, HEIGHT, WIDTH = 3, 224, 224
RESIZE_SHORT = 256
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype="float32")
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype="float32")


def preprocess(jpg_path, bin_path):
    "把一张 JPEG 处理成模型要的 NCHW float32 数据，返回写出的字节数。"
    with Image.open(jpg_path) as image:
        rgb = image.convert("RGB")
        width, height = rgb.size
        scale = RESIZE_SHORT / min(width, height)  # 等比缩放：短边到 256
        resized = rgb.resize(
            (round(width * scale), round(height * scale)), Image.BILINEAR
        )
        pixels = np.asarray(resized)
    offset_h = (pixels.shape[0] - HEIGHT) // 2  # 中心裁剪
    offset_w = (pixels.shape[1] - WIDTH) // 2
    cropped = pixels[offset_h : offset_h + HEIGHT, offset_w : offset_w + WIDTH, :]
    scaled = cropped.astype("float32") / 255.0  # 归一化到 0–1
    standardized = (scaled - IMAGENET_MEAN) / IMAGENET_STD  # 按训练配置标准化
    nchw = standardized.reshape(1, HEIGHT, WIDTH, CHANNELS).transpose(  # HWC -> NCHW
        [0, 3, 1, 2]
    )
    np.ascontiguousarray(nchw, dtype="float32").tofile(bin_path)
    return os.path.getsize(bin_path)


IMAGES = ["data/dog1_1024_683", "data/dog2_1024_683"]
SLOT_BYTES = CHANNELS * HEIGHT * WIDTH * 4

for stem in IMAGES:
    size = preprocess(stem + ".jpg", stem + ".bin")
    print(
        "%s.jpg -> %s.bin  %d 字节  %s"
        % (stem, stem, size, "✅" if size == SLOT_BYTES else "❌ 与预期不符")
    )
print("一张图应占 %d 字节（3 × 224 × 224 × 4）" % SLOT_BYTES)


### 7.3 主机侧参考结果

本实验**在主机侧用同一个 ONNX 模型算出一份参考结果**，再拿它去核对 NPU 的输出。这样做有三点考虑：

- 判据由本次运行自己建立。换一张图片、换一个模型，正文都不需要改动；
- 参考实现**自己完成一遍预处理，不调用 §7.2 的 `preprocess`**。两者是两份独立的实现，其中一份改错时，结果会分开；
- 比较的是 **Top-1 的类别标识，而不是逐个得分**。NPU 上的计算精度与主机侧的 FP32 不完全相同，得分会有差异；类别的次序在得分差距明显时不受影响。


In [ ]:
import importlib, subprocess, sys

try:
    importlib.import_module("onnxruntime")
except ImportError:
    print("onnxruntime 缺失，开始安装")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "onnxruntime"], check=False
    )

import onnxruntime as ort

# 多核服务器上 onnxruntime 默认会为每个物理核绑定线程，在容器里常因为 CPU 亲和性
# 受限而刷出大量告警。参考推理只跑两张图，限定线程数即可，同时把日志级别调到只报错误。
OPTIONS = ort.SessionOptions()
OPTIONS.intra_op_num_threads = 1
OPTIONS.inter_op_num_threads = 1
OPTIONS.log_severity_level = 3

SESSION = ort.InferenceSession(
    ONNX_PATH, sess_options=OPTIONS, providers=["CPUExecutionProvider"]
)
ORT_INPUT = SESSION.get_inputs()[0].name
print("ONNX 输入名 :", ORT_INPUT, SESSION.get_inputs()[0].shape)
print("ONNX 输出名 :", SESSION.get_outputs()[0].name, SESSION.get_outputs()[0].shape)


def reference_topk(jpg_path, k=5):
    """主机侧参考实现：独立完成一遍预处理，再用 onnxruntime 推理，返回 [(类别标识, 得分)]。"""
    with Image.open(jpg_path) as image:
        rgb = image.convert("RGB")
        w, h = rgb.size
        s = RESIZE_SHORT / min(w, h)
        arr = np.asarray(rgb.resize((round(w * s), round(h * s)), Image.BILINEAR))
    top = (arr.shape[0] - HEIGHT) // 2
    left = (arr.shape[1] - WIDTH) // 2
    patch = arr[top : top + HEIGHT, left : left + WIDTH, :].astype("float32") / 255.0
    nchw = ((patch - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1)[None]
    scores = SESSION.run(None, {ORT_INPUT: np.ascontiguousarray(nchw, "float32")})[0][0]
    order = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in order]


CLASS_NAME = {}
try:
    with open(LABEL_PATH, encoding="utf-8") as handle:
        for line_no, line in enumerate(handle):
            CLASS_NAME[line_no] = line.strip()
except OSError:
    print("⚠️  类别名称文件不可用，后面只显示类别标识")


def label(index):
    "类别标识转成可读的名称；名称文件缺失时退回标识本身。"
    return CLASS_NAME.get(index, "class %d" % index)


REFERENCE = [reference_topk(stem + ".jpg") for stem in IMAGES]
for stem, ranked in zip(IMAGES, REFERENCE):
    head = ranked[0]
    print(
        "%s.jpg  参考 Top-1 = %d（%s），得分 %.4f"
        % (stem, head[0], label(head[0]), head[1])
    )


## 8. 模型转换

转两个模型：一个固定 Batch = 1，一个动态 Batch。**除了那两个多出来的参数，命令完全一样。**

ResNet-18 有五十层，每次转换需要数分钟；两个模型都只在 `.om` 不存在时才转。ATC 的输出较长，只保留最后几行。


In [ ]:
import os, subprocess, time


def run_atc(name, extra):
    "调一次 atc，返回是否成功。命令模板见 §2。"
    target = "model/%s.om" % name
    if os.path.exists(target):
        print("✅ 已存在 %s（%.1f MB）\n" % (target, os.path.getsize(target) / 1048576))
        return True
    cmd = [
        "atc",
        "--model=" + ONNX_PATH,
        "--framework=5",
        "--output=model/" + name,
        "--soc_version=" + SOC_VERSION,
        "--input_format=NCHW",
        "--output_type=FP32",
    ] + extra
    print("$ " + " ".join(cmd))
    start = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start
    for line in (proc.stdout + proc.stderr).strip().splitlines()[-6:]:
        print("   " + line[:150])
    produced = os.path.exists(target)
    print(
        "✅ 生成 %s（%.1f MB），耗时 %.1f s"
        % (target, os.path.getsize(target) / 1048576, elapsed)
        if produced
        else "❌ 返回码 %d，未生成 .om" % proc.returncode
    )
    print()
    return produced


PROFILES = "1,2,4,8"
# 静态版按 ONNX 自带的形状转换；若 ATC 报形状不确定，
# 把 §7.3 打印出来的输入名填进 --input_shape，例如 ["--input_shape=x:1,3,224,224"]
STATIC_OK = run_atc("resnet18", [])
DYNAMIC_OK = run_atc(
    "resnet18_dynamic_batch",
    ["--input_shape=x:-1,3,224,224", "--dynamic_batch_size=" + PROFILES],
)


## 9. 程序实现

程序分六段写入 `src_model/acl_model.cpp`。**它按模型描述工作，不针对某一个模型**：输入的个数、每个的字节数与名字全部来自 `aclmdlDesc`，因此固定 Batch 与动态 Batch 两个模型由同一个二进制处理。

程序实现流程：

<img src="images/07.06_infer_flow.png" alt="基本的模型推理流程" width="600">

可以看出本实验取的是哪几条分支：**执行方式选同步**，因此不出现 callback 与 `aclrtSynchronizeStream`；**绿色的可选步骤只用了「设置动态 batch」一项**，对应 §5 的批大小选择；末尾释放的次序与 §4.3 的两条要求一致。

### 9.1 头文件、常量与错误检查

只 `#include` 了 `acl/acl.h`。模型管理接口与 Runtime 接口在同一个头文件里，链接时要多一个模型管理库（§6 已经查过）。


In [ ]:
%%writefile  src_model/acl_model.cpp
/**
 * Parallel Computing, Chapter 7, Lab 7: Model Inference
 *
 * A model is an operator graph that has already been compiled for this
 * processor. This program loads a ResNet-18 offline model, asks it what it
 * expects, classifies real images with it, and reports where the wall clock
 * goes: loading, the first inference, the steady-state inference, and the
 * transfers on either side.
 *
 * The program is written against the model description rather than against a
 * particular model: the number of inputs, their sizes, their names and their
 * shapes all come from aclmdlDesc, so the fixed-batch model and the dynamic
 * batch model are both handled by the same binary.
 *
 * Usage: acl_model <mode> [arguments]
 *   info  <om>                              describe the model
 *   infer <om> <repeat> <bin> [bin ...]     classify each image, split time up
 *   mem   <om>                              system-managed vs caller-managed
 *   batch <om> <p,p,...> <repeat> <bin> ... sweep the batch profiles
 */
#include <cstdint>  // int64_t, uint32_t, uint64_t
#include <cstdio>   // std::printf, std::fopen
#include <cstdlib>  // std::atoi, std::strtoll
#include <cstring>  // std::strcmp, std::memcpy
#include <ctime>    // clock_gettime, timespec
#include <string>   // std::string
#include <vector>   // std::vector

#include "acl/acl.h"  // Runtime and model management APIs

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kWarmupRuns = 3;
constexpr int kDefaultRepeat = 20;
constexpr int kTopK = 5;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 9.2 计时、读文件与两种加载

`Model` 结构把一个已加载模型的全部资源集中管理。`LoadSystemManaged` 与 `LoadUserManaged` 分别对应 §3 的两类接口，后者的注释中标出了 §3.2 那处次序。


In [ ]:
%%writefile -a src_model/acl_model.cpp
// Returns a monotonic timestamp in milliseconds, for the host-side clock.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Reads a whole file into a host buffer.
int ReadFile(const char* path, std::vector<char>* content) {
  FILE* handle = std::fopen(path, "rb");
  if (handle == nullptr) {
    std::fprintf(stderr, "[ERR] api=fopen code=- msg=cannot open %s\n", path);
    return ACL_ERROR_INVALID_PARAM;
  }
  std::fseek(handle, 0, SEEK_END);
  const long size = std::ftell(handle);
  std::fseek(handle, 0, SEEK_SET);
  content->assign(static_cast<size_t>(size), 0);
  const size_t got =
      std::fread(content->data(), 1, static_cast<size_t>(size), handle);
  std::fclose(handle);
  if (got != static_cast<size_t>(size)) {
    std::fprintf(stderr, "[ERR] api=fread code=- msg=short read on %s\n", path);
    return ACL_ERROR_INVALID_PARAM;
  }
  return ACL_SUCCESS;
}

// Everything one loaded model needs: the id, the description, and the two
// datasets with the device buffers they point at.
struct Model {
  uint32_t id;
  aclmdlDesc* desc;
  aclmdlDataset* input;
  aclmdlDataset* output;
  void* work;
  void* weight;
  bool user_memory;
};

// Loads a model and lets the system manage the memory it runs in. This form
// works for every model, including one whose input shape is not fixed.
int LoadSystemManaged(const char* path, Model* model) {
  model->work = nullptr;
  model->weight = nullptr;
  model->user_memory = false;
  ACL_CHECK(aclmdlLoadFromFile(path, &model->id));
  return ACL_SUCCESS;
}

// Loads a model and manages the memory it runs in from the caller side. Two
// blocks are needed: working memory for the temporaries of one execution, and
// weight memory for the parameters. Note the order in which they appear: the
// query reports (work, weight) and the load takes (work ptr, work size,
// weight ptr, weight size).
int LoadUserManaged(const char* path, Model* model, size_t* work_size,
                    size_t* weight_size) {
  ACL_CHECK(aclmdlQuerySize(path, work_size, weight_size));
  ACL_CHECK(aclrtMalloc(&model->work, *work_size, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(
      aclrtMalloc(&model->weight, *weight_size, ACL_MEM_MALLOC_HUGE_FIRST));
  ACL_CHECK(aclmdlLoadFromFileWithMem(path, &model->id, model->work, *work_size,
                                      model->weight, *weight_size));
  model->user_memory = true;
  return ACL_SUCCESS;
}


### 9.3 数据组织：建、填、读、放

`CreateDataset` 实现流程：

<img src="images/07.06_prepare_dataset.png" alt="模型执行的输入输出数据结构的准备流程" width="500">

图中灰色框里的四步在本程序里就是一个循环：问大小、申请内存、建 DataBuffer、挂进 Dataset。右侧绿色的「获取输入/输出的名称」在本实验里不是可选项，§5.2 的批大小选择器就是靠名称找到的。

这一段是本程序**与模型无关**的部分：

- `CreateDataset` 按 `aclmdlGetNumInputs` / `aclmdlGetInputSizeByIndex` 分配，不写死；**批大小选择器那一个缓冲区跳过清零**。
- `FillInput` 把图片逐张填进输入缓冲区的各个位置，图片数量不足时循环取用，因此两张图可以填满批大小为 8 的输入。
- `VerifyInput` 把输入缓冲区从设备回读，逐个位置与文件比对。**写进去与真的写进去是两件事**，这一步把它变成一次测量。
- `PrintTopK` 从一张图的 1000 个得分里挑出最高的几个。**一张图的得分是连续的一段**，所以取第 $i$ 张图只是把指针挪 $i \times 1000$ 个 float。
- `DestroyDataset` 与 `UnloadModel` 把 §4.3 的两条释放次序固定下来，所有释放都经过它们。


In [ ]:
%%writefile -a src_model/acl_model.cpp
// Returns the index of the profile selector input, or the number of inputs
// when this model does not have one. A dynamic batch model carries one extra
// input under a fixed name, and the runtime is its only writer.
size_t FindSelector(aclmdlDesc* desc) {
  size_t index = 0;
  if (aclmdlGetInputIndexByName(desc, ACL_DYNAMIC_TENSOR_NAME, &index) !=
      ACL_SUCCESS) {
    return aclmdlGetNumInputs(desc);
  }
  return index;
}

// Builds one dataset with one device buffer per model input or output. The
// sizes come from the model description, so nothing here is model specific.
// The profile selector is the one buffer left untouched: writing to it is not
// allowed.
int CreateDataset(aclmdlDesc* desc, bool is_input, aclmdlDataset** dataset) {
  *dataset = aclmdlCreateDataset();
  if (*dataset == nullptr) {
    std::fprintf(stderr, "[ERR] api=aclmdlCreateDataset code=- msg=null\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  const size_t count =
      is_input ? aclmdlGetNumInputs(desc) : aclmdlGetNumOutputs(desc);
  const size_t selector = is_input ? FindSelector(desc) : count;
  for (size_t i = 0; i < count; ++i) {
    const size_t bytes = is_input ? aclmdlGetInputSizeByIndex(desc, i)
                                  : aclmdlGetOutputSizeByIndex(desc, i);
    void* device = nullptr;
    ACL_CHECK(aclrtMalloc(&device, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    if (i != selector) {
      ACL_CHECK(aclrtMemset(device, bytes, 0, bytes));
    }
    aclDataBuffer* buffer = aclCreateDataBuffer(device, bytes);
    if (buffer == nullptr) {
      std::fprintf(stderr, "[ERR] api=aclCreateDataBuffer code=- msg=null\n");
      return ACL_ERROR_INVALID_PARAM;
    }
    ACL_CHECK(aclmdlAddDatasetBuffer(*dataset, buffer));
  }
  return ACL_SUCCESS;
}

// Releases a dataset. The order is fixed: take the address out of the buffer
// first, then free the memory, then destroy the buffer, then the dataset.
// Destroying the buffer first would lose the only handle on the address.
void DestroyDataset(aclmdlDataset* dataset) {
  if (dataset == nullptr) {
    return;
  }
  for (size_t i = 0; i < aclmdlGetDatasetNumBuffers(dataset); ++i) {
    aclDataBuffer* buffer = aclmdlGetDatasetBuffer(dataset, i);
    void* address = aclGetDataBufferAddr(buffer);
    (void)aclrtFree(address);
    (void)aclDestroyDataBuffer(buffer);
  }
  (void)aclmdlDestroyDataset(dataset);
}

// Unloads a model. The model must be unloaded before the memory it runs in is
// released, otherwise the runtime is left pointing at freed memory.
void UnloadModel(Model* model) {
  DestroyDataset(model->output);
  DestroyDataset(model->input);
  model->output = nullptr;
  model->input = nullptr;
  (void)aclmdlUnload(model->id);
  if (model->desc != nullptr) {
    (void)aclmdlDestroyDesc(model->desc);
    model->desc = nullptr;
  }
  if (model->work != nullptr) {
    (void)aclrtFree(model->work);
    model->work = nullptr;
  }
  if (model->weight != nullptr) {
    (void)aclrtFree(model->weight);
    model->weight = nullptr;
  }
}

// Copies one image into each slot of the input buffer, cycling through the
// file list when the batch is wider than the number of files available.
int FillInput(const std::vector<std::string>& files, void* device,
              size_t slot_bytes, size_t slots) {
  for (size_t slot = 0; slot < slots; ++slot) {
    const std::string& name = files[slot % files.size()];
    std::vector<char> host;
    ACL_CHECK(ReadFile(name.c_str(), &host));
    if (host.size() != slot_bytes) {
      std::fprintf(stderr,
                   "[ERR] api=FillInput code=- msg=%s holds %zu bytes but one "
                   "image occupies %zu\n",
                   name.c_str(), host.size(), slot_bytes);
      return ACL_ERROR_INVALID_PARAM;
    }
    ACL_CHECK(aclrtMemcpy(static_cast<char*>(device) + slot * slot_bytes,
                          slot_bytes, host.data(), slot_bytes,
                          ACL_MEMCPY_HOST_TO_DEVICE));
  }
  return ACL_SUCCESS;
}

// Reads the input buffer back from the device and compares every slot with the
// file it was filled from. Returns the index of the first slot that differs, or
// `slots` when all of them match.
size_t VerifyInput(const std::vector<std::string>& files, const void* device,
                   size_t slot_bytes, size_t slots) {
  std::vector<char> back(slot_bytes);
  std::vector<char> host;
  for (size_t slot = 0; slot < slots; ++slot) {
    if (ReadFile(files[slot % files.size()].c_str(), &host) != ACL_SUCCESS) {
      return slot;
    }
    if (aclrtMemcpy(back.data(), slot_bytes,
                    static_cast<const char*>(device) + slot * slot_bytes,
                    slot_bytes, ACL_MEMCPY_DEVICE_TO_HOST) != ACL_SUCCESS) {
      return slot;
    }
    if (std::memcmp(back.data(), host.data(), slot_bytes) != 0) {
      return slot;
    }
  }
  return slots;
}

// Prints the k highest scores of one image, most confident first. The scores
// of one image are contiguous, so one image is a window into the output.
void PrintTopK(long long batch, size_t image, const float* scores,
               size_t classes, int k) {
  std::vector<char> taken(classes, 0);
  for (int rank = 1; rank <= k; ++rank) {
    size_t best = classes;
    for (size_t i = 0; i < classes; ++i) {
      if (taken[i] == 0 && (best == classes || scores[i] > scores[best])) {
        best = i;
      }
    }
    if (best == classes) {
      return;
    }
    taken[best] = 1;
    std::printf("[TOP] batch=%lld image=%zu rank=%d index=%zu score=%.6f\n",
                batch, image, rank, best, static_cast<double>(scores[best]));
  }
}


### 9.4 两个只读的模式：读取规格与查询内存

`RunInfo` 读取并打印模型的输入输出规格：输入输出的个数、名字、字节数与维度，是否存在动态 Batch 的批大小选择器输入，以及支持哪些批大小。维度中出现 `-1` 表示这一维在构建时没有固定。

`RunMem` 把同一个模型加载两次，一次由系统管理运行内存、一次由调用者管理，并把 `aclmdlQuerySize` 报出的两块大小打印出来。**查询失败不作为程序出错处理**：§3.1 提到有一类模型查不出内存大小，程序记录返回码后跳过这一次加载。


In [ ]:
%%writefile -a src_model/acl_model.cpp
// Prints one line per input and per output: the name the model knows it by,
// the shape it declares, and how many bytes the buffer takes. A dimension
// printed as -1 is one that was left open at build time.
int RunInfo(const char* path) {
  Model model = {};
  ACL_CHECK(LoadSystemManaged(path, &model));
  model.desc = aclmdlCreateDesc();
  ACL_CHECK(aclmdlGetDesc(model.desc, model.id));

  const size_t inputs = aclmdlGetNumInputs(model.desc);
  const size_t outputs = aclmdlGetNumOutputs(model.desc);
  std::printf("[MODEL] path=%s inputs=%zu outputs=%zu\n", path, inputs,
              outputs);

  for (size_t i = 0; i < inputs; ++i) {
    aclmdlIODims dims = {};
    ACL_CHECK(aclmdlGetInputDims(model.desc, i, &dims));
    std::printf("[MODELIN] index=%zu name=%s bytes=%zu dims=", i,
                aclmdlGetInputNameByIndex(model.desc, i),
                aclmdlGetInputSizeByIndex(model.desc, i));
    for (size_t d = 0; d < dims.dimCount; ++d) {
      std::printf("%s%lld", (d == 0) ? "" : "x",
                  static_cast<long long>(dims.dims[d]));
    }
    std::printf("\n");
  }
  for (size_t i = 0; i < outputs; ++i) {
    aclmdlIODims dims = {};
    ACL_CHECK(aclmdlGetOutputDims(model.desc, i, &dims));
    std::printf("[MODELOUT] index=%zu name=%s bytes=%zu dims=", i,
                aclmdlGetOutputNameByIndex(model.desc, i),
                aclmdlGetOutputSizeByIndex(model.desc, i));
    for (size_t d = 0; d < dims.dimCount; ++d) {
      std::printf("%s%lld", (d == 0) ? "" : "x",
                  static_cast<long long>(dims.dims[d]));
    }
    std::printf("\n");
  }

  size_t selector = 0;
  const int status =
      aclmdlGetInputIndexByName(model.desc, ACL_DYNAMIC_TENSOR_NAME, &selector);
  std::printf("[DYNAMIC] status=%d index=%zu supported=%s\n", status, selector,
              (status == ACL_SUCCESS) ? "yes" : "no");

  aclmdlBatch gears = {};
  const int gear_status = aclmdlGetDynamicBatch(model.desc, &gears);
  std::printf("[GEARS] status=%d count=%zu list=", gear_status,
              gears.batchCount);
  for (size_t i = 0; i < gears.batchCount; ++i) {
    std::printf("%s%llu", (i == 0) ? "" : ",",
                static_cast<unsigned long long>(gears.batch[i]));
  }
  std::printf("%s\n", (gears.batchCount == 0) ? "-" : "");

  UnloadModel(&model);
  std::printf("[RESULT] PASS\n");
  return ACL_SUCCESS;
}

// Loads the same model twice, once letting the system manage the memory it
// runs in and once managing it from the caller side, and reports what each
// costs. The query reports the two sizes the second form has to allocate; a
// model whose input shape is not fixed cannot report them, and the program
// records the return code rather than treating it as a failure.
int RunMem(const char* path) {
  const double system_start = GetTimeMs();
  Model system_model = {};
  ACL_CHECK(LoadSystemManaged(path, &system_model));
  const double system_ms = GetTimeMs() - system_start;
  std::printf("[LOAD] owner=system load_ms=%.4f work_bytes=- weight_bytes=-\n",
              system_ms);
  UnloadModel(&system_model);

  size_t work_size = 0;
  size_t weight_size = 0;
  const double query_start = GetTimeMs();
  const int query = aclmdlQuerySize(path, &work_size, &weight_size);
  const double query_ms = GetTimeMs() - query_start;
  if (query != ACL_SUCCESS) {
    std::printf(
        "[LOAD] owner=user load_ms=- work_bytes=- weight_bytes=- "
        "query_status=%d\n",
        query);
    std::printf("[RESULT] PASS\n");
    return ACL_SUCCESS;
  }

  const double user_start = GetTimeMs();
  Model user_model = {};
  ACL_CHECK(LoadUserManaged(path, &user_model, &work_size, &weight_size));
  const double user_ms = GetTimeMs() - user_start;
  std::printf(
      "[LOAD] owner=user load_ms=%.4f work_bytes=%zu weight_bytes=%zu "
      "query_ms=%.4f query_status=0\n",
      user_ms, work_size, weight_size, query_ms);
  UnloadModel(&user_model);

  std::printf("[RESULT] PASS\n");
  return ACL_SUCCESS;
}


### 9.5 分类与阶段分解

`RunInfer` 逐张处理图片：传入、执行、取回、报出 Top-5。计时上有三点要求：

- **加载计时从 `aclmdlLoadFromFile` 之前开始，到 Dataset 建立完成为止**：建立 Dataset 同样要申请设备内存，属于模型可用之前必须完成的工作；
- **首次推理单独计时**，不与后面的平均值混在一起；
- **稳态推理之前另有 `kWarmupRuns` 轮预热**，使首次执行的一次性开销不计入平均值。


In [ ]:
%%writefile -a src_model/acl_model.cpp
// Loads the model, classifies each image in turn, and splits the wall clock
// into the parts that behave differently: loading, the first inference, the
// steady-state inference, and the two transfers.
int RunInfer(const char* path, int repeat,
             const std::vector<std::string>& files) {
  const double load_start = GetTimeMs();
  Model model = {};
  ACL_CHECK(LoadSystemManaged(path, &model));
  model.desc = aclmdlCreateDesc();
  ACL_CHECK(aclmdlGetDesc(model.desc, model.id));
  ACL_CHECK(CreateDataset(model.desc, true, &model.input));
  ACL_CHECK(CreateDataset(model.desc, false, &model.output));
  std::printf("[LOADTIME] load_ms=%.4f\n", GetTimeMs() - load_start);

  aclDataBuffer* in_buffer = aclmdlGetDatasetBuffer(model.input, 0);
  aclDataBuffer* out_buffer = aclmdlGetDatasetBuffer(model.output, 0);
  const size_t in_bytes = aclGetDataBufferSizeV2(in_buffer);
  const size_t out_bytes = aclGetDataBufferSizeV2(out_buffer);
  aclmdlIODims out_dims = {};
  ACL_CHECK(aclmdlGetOutputDims(model.desc, 0, &out_dims));
  const size_t classes =
      (out_dims.dimCount >= 2)
          ? static_cast<size_t>(out_dims.dims[out_dims.dimCount - 1])
          : out_bytes / sizeof(float);

  std::vector<char> host_out(out_bytes);
  for (size_t index = 0; index < files.size(); ++index) {
    std::vector<char> host_in;
    ACL_CHECK(ReadFile(files[index].c_str(), &host_in));
    if (host_in.size() != in_bytes) {
      std::fprintf(stderr,
                   "[ERR] api=RunInfer code=- msg=%s holds %zu bytes but the "
                   "model input takes %zu\n",
                   files[index].c_str(), host_in.size(), in_bytes);
      return ACL_ERROR_INVALID_PARAM;
    }
    std::printf("[IMAGE] image=%zu file=%s\n", index, files[index].c_str());

    const double h2d_start = GetTimeMs();
    ACL_CHECK(aclrtMemcpy(aclGetDataBufferAddr(in_buffer), in_bytes,
                          host_in.data(), in_bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    const double h2d_ms = GetTimeMs() - h2d_start;

    const double first_start = GetTimeMs();
    ACL_CHECK(aclmdlExecute(model.id, model.input, model.output));
    const double first_ms = GetTimeMs() - first_start;

    for (int warm = 0; warm < kWarmupRuns; ++warm) {
      ACL_CHECK(aclmdlExecute(model.id, model.input, model.output));
    }
    const double steady_start = GetTimeMs();
    for (int run = 0; run < repeat; ++run) {
      ACL_CHECK(aclmdlExecute(model.id, model.input, model.output));
    }
    const double steady_ms = (GetTimeMs() - steady_start) / repeat;

    const double d2h_start = GetTimeMs();
    ACL_CHECK(aclrtMemcpy(host_out.data(), out_bytes,
                          aclGetDataBufferAddr(out_buffer), out_bytes,
                          ACL_MEMCPY_DEVICE_TO_HOST));
    const double d2h_ms = GetTimeMs() - d2h_start;

    const double e2e_ms = h2d_ms + steady_ms + d2h_ms;
    std::printf(
        "[STAGE] image=%zu first_ms=%.4f steady_ms=%.4f h2d_ms=%.4f "
        "d2h_ms=%.4f in_bytes=%zu out_bytes=%zu repeat=%d e2e_ms=%.4f "
        "fps=%.2f\n",
        index, first_ms, steady_ms, h2d_ms, d2h_ms, in_bytes, out_bytes, repeat,
        e2e_ms, 1000.0 / e2e_ms);
    PrintTopK(1, index, reinterpret_cast<const float*>(host_out.data()),
              classes, kTopK);
  }

  UnloadModel(&model);
  std::printf("[RESULT] PASS\n");
  return ACL_SUCCESS;
}


### 9.6 动态 Batch 与主程序

`RunBatch` 对每一个批大小执行相同的步骤：填入图片、回读校验、设置批大小、预热、计时、报出每张图的 Top-1。四点需要说明：

- **图像输入取的是批大小选择器之外的那个索引**，不假定它在 0 号位（§4.2）；
- **批大小选择器对应的内存全程没有被写入**，只有 `aclmdlSetDynamicBatchSize` 访问它（§5.2 第 2 条）；
- 一张图的字节数由图片文件本身决定，程序据此校验输入缓冲区是否足以容纳最大的批大小；
- 每一个批大小都调用 `aclmdlGetCurOutputDims`，把**这一次实际的输出维度**与缓冲区大小并排打印（§5.3）。


In [ ]:
%%writefile -a src_model/acl_model.cpp
// Sweeps the dynamic batch profiles. Two rules govern this mode: the value set
// must be one of the profiles chosen at build time, and the buffer that holds
// the profile selector must not be written by the caller.
int RunBatch(const char* path, const char* profiles, int repeat,
             const std::vector<std::string>& files) {
  Model model = {};
  ACL_CHECK(LoadSystemManaged(path, &model));
  model.desc = aclmdlCreateDesc();
  ACL_CHECK(aclmdlGetDesc(model.desc, model.id));
  ACL_CHECK(CreateDataset(model.desc, true, &model.input));
  ACL_CHECK(CreateDataset(model.desc, false, &model.output));

  size_t selector = 0;
  ACL_CHECK(aclmdlGetInputIndexByName(model.desc, ACL_DYNAMIC_TENSOR_NAME,
                                      &selector));
  aclmdlBatch gears = {};
  ACL_CHECK(aclmdlGetDynamicBatch(model.desc, &gears));
  uint64_t widest = 1;
  for (size_t i = 0; i < gears.batchCount; ++i) {
    widest = (gears.batch[i] > widest) ? gears.batch[i] : widest;
  }
  std::printf("[CFG] mode=batch selector=%zu widest=%llu repeat=%d\n", selector,
              static_cast<unsigned long long>(widest), repeat);

  // 图像输入是除批大小选择器之外的那一个。不能假定它在索引 0：
  // 输入的次序由 ATC 决定，选择器不一定排在最后（§4.2）。
  size_t image_index = 0;
  for (size_t i = 0; i < aclmdlGetNumInputs(model.desc); ++i) {
    if (i != selector) {
      image_index = i;
      break;
    }
  }
  aclDataBuffer* in_buffer = aclmdlGetDatasetBuffer(model.input, image_index);
  aclDataBuffer* out_buffer = aclmdlGetDatasetBuffer(model.output, 0);
  const size_t in_bytes = aclGetDataBufferSizeV2(in_buffer);
  const size_t out_bytes = aclGetDataBufferSizeV2(out_buffer);
  aclmdlIODims out_dims = {};
  ACL_CHECK(aclmdlGetOutputDims(model.desc, 0, &out_dims));
  const size_t classes =
      (out_dims.dimCount >= 2)
          ? static_cast<size_t>(out_dims.dims[out_dims.dimCount - 1])
          : out_bytes / (sizeof(float) * widest);

  std::vector<char> probe;
  ACL_CHECK(ReadFile(files[0].c_str(), &probe));
  const size_t slot_bytes = probe.size();
  if (slot_bytes * widest > in_bytes) {
    std::fprintf(stderr,
                 "[ERR] api=RunBatch code=- msg=one image is %zu bytes and the "
                 "widest profile is %llu, but the input buffer holds %zu\n",
                 slot_bytes, static_cast<unsigned long long>(widest), in_bytes);
    return ACL_ERROR_INVALID_PARAM;
  }

  bool all_passed = true;
  std::vector<char> host_out(out_bytes);
  const char* cursor = profiles;
  while (*cursor != '\0') {
    char* stop = nullptr;
    const long long batch = std::strtoll(cursor, &stop, 10);
    if (stop == cursor || batch <= 0) {
      std::fprintf(stderr, "[ERR] api=RunBatch code=- msg=bad profile list\n");
      return ACL_ERROR_INVALID_PARAM;
    }
    cursor = (*stop == ',') ? stop + 1 : stop;

    // 次序与官方样例一致：先把输入数据写入设备，再设置这一次的批大小。
    ACL_CHECK(FillInput(files, aclGetDataBufferAddr(in_buffer), slot_bytes,
                        static_cast<size_t>(batch)));
    const size_t bad = VerifyInput(files, aclGetDataBufferAddr(in_buffer),
                                   slot_bytes, static_cast<size_t>(batch));
    std::printf("[FILL] batch=%lld image_index=%zu slots=%lld matched=%s "
                "first_bad=%zu\n",
                batch, image_index, batch,
                (bad == static_cast<size_t>(batch)) ? "all" : "no", bad);

    const int set = aclmdlSetDynamicBatchSize(model.id, model.input, selector,
                                              static_cast<uint64_t>(batch));
    if (set != ACL_SUCCESS) {
      std::printf("[BATCH] batch=%lld status=%d e2e_ms=- fps=-\n", batch, set);
      all_passed = false;
      continue;
    }

    for (int warm = 0; warm < kWarmupRuns; ++warm) {
      ACL_CHECK(aclmdlExecute(model.id, model.input, model.output));
    }
    const double start = GetTimeMs();
    for (int run = 0; run < repeat; ++run) {
      ACL_CHECK(aclmdlExecute(model.id, model.input, model.output));
    }
    const double e2e_ms = (GetTimeMs() - start) / repeat;

    ACL_CHECK(aclrtMemcpy(host_out.data(), out_bytes,
                          aclGetDataBufferAddr(out_buffer), out_bytes,
                          ACL_MEMCPY_DEVICE_TO_HOST));
    aclmdlIODims current = {};
    const int cur = aclmdlGetCurOutputDims(model.desc, 0, &current);
    std::printf(
        "[BATCH] batch=%lld status=0 e2e_ms=%.4f fps=%.2f per_sample_ms=%.4f "
        "out_bytes=%zu cur_status=%d cur_dim0=%lld\n",
        batch, e2e_ms, static_cast<double>(batch) * 1000.0 / e2e_ms,
        e2e_ms / static_cast<double>(batch), out_bytes, cur,
        (cur == ACL_SUCCESS && current.dimCount > 0)
            ? static_cast<long long>(current.dims[0])
            : -1);
    const float* scores = reinterpret_cast<const float*>(host_out.data());
    for (long long image = 0; image < batch; ++image) {
      PrintTopK(batch, static_cast<size_t>(image),
                scores + static_cast<size_t>(image) * classes, classes, 1);
    }
  }

  UnloadModel(&model);
  std::printf("[RESULT] %s\n", all_passed ? "PASS" : "FAIL");
  return all_passed ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}

int Dispatch(int argc, char** argv) {
  const char* mode = (argc > 1) ? argv[1] : "info";
  const char* path = (argc > 2) ? argv[2] : "model/resnet18.om";
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));
  aclrtContext context = nullptr;
  ACL_CHECK(aclrtCreateContext(&context, kDeviceId));
  ACL_CHECK(aclrtSetCurrentContext(context));
  std::printf("[ENV] soc=%s\n", aclrtGetSocName());

  std::vector<std::string> files;
  int status = ACL_SUCCESS;
  if (std::strcmp(mode, "info") == 0) {
    status = RunInfo(path);
  } else if (std::strcmp(mode, "mem") == 0) {
    status = RunMem(path);
  } else if (std::strcmp(mode, "infer") == 0) {
    const int repeat = (argc > 3) ? std::atoi(argv[3]) : kDefaultRepeat;
    for (int i = 4; i < argc; ++i) {
      files.push_back(argv[i]);
    }
    status =
        files.empty() ? ACL_ERROR_INVALID_PARAM : RunInfer(path, repeat, files);
  } else if (std::strcmp(mode, "batch") == 0) {
    const char* profiles = (argc > 3) ? argv[3] : "1,2,4,8";
    const int repeat = (argc > 4) ? std::atoi(argv[4]) : kDefaultRepeat;
    for (int i = 5; i < argc; ++i) {
      files.push_back(argv[i]);
    }
    status = files.empty() ? ACL_ERROR_INVALID_PARAM
                           : RunBatch(path, profiles, repeat, files);
  } else {
    std::fprintf(stderr, "[ERR] api=main code=- msg=unknown mode %s\n", mode);
    status = ACL_ERROR_INVALID_PARAM;
  }

  (void)aclrtDestroyContext(context);
  (void)aclrtResetDevice(kDeviceId);
  (void)aclFinalize();
  return status;
}

int main(int argc, char** argv) {
  return (Dispatch(argc, argv) == ACL_SUCCESS) ? 0 : 1;
}


## 10. 编译与运行

编译命令只用 `g++`，链接 Runtime 库与模型管理库两个动态库。


In [ ]:
import os, subprocess

cmd = (
    ["g++", "src_model/acl_model.cpp", "-std=c++17", "-O2", "-Wall", "-Wextra"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", "src_model/acl_model"]
)
print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print((proc.stdout + proc.stderr).strip() or "✅ 编译通过")


四种模式各运行一次，`info` 与 `mem` 对两个模型各运行一遍。**固定 Batch 与动态 Batch 的输入个数不同**，对应 §5.2 的批大小选择器输入。


In [ ]:
import subprocess

REPEAT = 20
BINS = [stem + ".bin" for stem in IMAGES]
EXE = "./src_model/acl_model"
out = {}

runs = []
if STATIC_OK:
    runs += [
        ("info", [EXE, "info", "model/resnet18.om"]),
        ("mem", [EXE, "mem", "model/resnet18.om"]),
        ("infer", [EXE, "infer", "model/resnet18.om", str(REPEAT)] + BINS),
    ]
if DYNAMIC_OK:
    runs += [
        ("info_dyn", [EXE, "info", "model/resnet18_dynamic_batch.om"]),
        ("mem_dyn", [EXE, "mem", "model/resnet18_dynamic_batch.om"]),
        (
            "batch",
            [EXE, "batch", "model/resnet18_dynamic_batch.om", PROFILES, str(REPEAT)]
            + BINS,
        ),
    ]

for name, cmd in runs:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    out[name] = proc.stdout
    tail = proc.stdout.strip().splitlines()[-1:] or ["(无输出)"]
    print(f"=== {name:9s} 退出码 {proc.returncode}  {tail[0]}")
    if proc.returncode != 0:
        print(proc.stderr.strip()[:800])


记录行的格式是一个方括号标签，后面跟若干个 `键=值`。


In [ ]:
NUMERIC = {
    "inputs",
    "outputs",
    "index",
    "image",
    "rank",
    "bytes",
    "status",
    "count",
    "load_ms",
    "first_ms",
    "steady_ms",
    "h2d_ms",
    "d2h_ms",
    "in_bytes",
    "out_bytes",
    "repeat",
    "slots",
    "first_bad",
    "image_index",
    "e2e_ms",
    "fps",
    "score",
    "batch",
    "per_sample_ms",
    "cur_status",
    "cur_dim0",
    "work_bytes",
    "weight_bytes",
    "query_ms",
    "query_status",
}


def parse(text, tag):
    "把带某个标签的所有记录行解析成字典列表。"
    rows = []
    for line in text.splitlines():
        if not line.startswith(tag + " "):
            continue
        item = {}
        for token in line[len(tag) + 1 :].split():
            if "=" not in token:
                continue
            key, value = token.split("=", 1)
            if key in NUMERIC:
                # 值为 "-" 表示这一行没有这个字段，直接不收，
                # 这样后面用 in 判断就不会取到一个不能参与计算的字符串
                if value != "-":
                    item[key] = float(value)
            else:
                item[key] = value
        rows.append(item)
    return rows


images = parse(out.get("infer", ""), "[IMAGE]")
stage = parse(out.get("infer", ""), "[STAGE]")
tops = parse(out.get("infer", ""), "[TOP]")
loadtime = parse(out.get("infer", ""), "[LOADTIME]")
loads = parse(out.get("mem", ""), "[LOAD]")
loads_dyn = parse(out.get("mem_dyn", ""), "[LOAD]")
batch = parse(out.get("batch", ""), "[BATCH]")
batch_tops = parse(out.get("batch", ""), "[TOP]")
fills = parse(out.get("batch", ""), "[FILL]")
print(
    "解析到：image %d 行，stage %d 行，top %d 行，load %d + %d 行，batch %d 行，fill %d 行"
    % (
        len(images),
        len(stage),
        len(tops),
        len(loads),
        len(loads_dyn),
        len(batch),
        len(fills),
    )
)


## 11. 模型的输入输出规格

两个模型的规格并排打印。**除了多出来的批大小选择器输入与第一维的 `-1`，其余完全相同**：两者出自同一个 ONNX，只是 ATC 的参数不同。

三个数字可以互相印证：输入 602112 字节对应一张 224×224 图像的三个 float32 通道；输出 4000 字节对应 1000 个类别的得分；动态模型的这两个数字各为最大批大小 8 的倍数。


In [ ]:
for heading, key in (
    ("固定 Batch = 1", "info"),
    ("动态 Batch " + PROFILES, "info_dyn"),
):
    text = out.get(key, "")
    if not text:
        continue
    print("=" * 72)
    print(heading)
    print("=" * 72)
    head = parse(text, "[MODEL]")
    if head:
        print(
            "输入 %d 个，输出 %d 个" % (int(head[0]["inputs"]), int(head[0]["outputs"]))
        )
    for row in parse(text, "[MODELIN]"):
        print(
            "  输入 %d  name=%-26s bytes=%-9d dims=%s"
            % (int(row["index"]), row["name"], int(row["bytes"]), row["dims"])
        )
    for row in parse(text, "[MODELOUT]"):
        print(
            "  输出 %d  name=%-26s bytes=%-9d dims=%s"
            % (int(row["index"]), row["name"], int(row["bytes"]), row["dims"])
        )
    for row in parse(text, "[DYNAMIC]"):
        print(
            "  批大小选择器：%s"
            % (
                "有，index=%d" % int(row["index"])
                if row["supported"] == "yes"
                else "无（返回码 %d）" % int(row["status"])
            )
        )
    for row in parse(text, "[GEARS]"):
        print("  支持的批大小：%s" % row.get("list", "-"))
    print()


## 12. 分类结果

两张图各自的 Top-5，以及与 §7.3 主机侧参考的核对。**判据是 Top-1 的类别标识是否与参考一致**：一致说明预处理、模型转换、数据组织与输出读取这一整条通路都正确。

后四名不参与判据。它们是模型认为次可能的类别，与图片内容和训练数据有关；NPU 与主机侧的计算精度不同，这几名之间的次序本来就可能有出入。


In [ ]:
if not tops:
    print("没有解析到 [TOP] 行，请先确认 §10 的 infer 已运行成功。")
else:
    passed = 0
    for row in images:
        index = int(row["image"])
        ranked = sorted(
            (r for r in tops if int(r["image"]) == index), key=lambda r: r["rank"]
        )
        reference = REFERENCE[index] if index < len(REFERENCE) else []
        print("=" * 78)
        print("图片 %d：%s" % (index, row["file"]))
        print("=" * 78)
        print("  %-6s %-8s %-12s %s" % ("排名", "类别标识", "NPU 得分", "类别名称"))
        for r in ranked:
            print(
                "  %-6d %-8d %-12.6f %s"
                % (int(r["rank"]), int(r["index"]), r["score"], label(int(r["index"])))
            )
        npu_top1 = int(ranked[0]["index"]) if ranked else -1
        ref_top1 = reference[0][0] if reference else -1
        ok = npu_top1 == ref_top1
        passed += int(ok)
        print(
            "  主机参考 Top-1 = %d（%s），NPU Top-1 = %d —— %s\n"
            % (ref_top1, label(ref_top1), npu_top1, "PASS" if ok else "FAIL")
        )
    print("两张图的 Top-1 与主机参考核对：%d / %d 一致" % (passed, len(images)))


把分类结果与图片放在一起看。左边是原始照片，右边是 Top-5 的概率；**与主机侧参考一致的那一项用深色标出**。

概率由 Top-5 的得分做一次 Softmax 得到。模型最后一层是全连接，输出的是未归一化的得分；这一步只影响显示，不改变名次。


In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False


def softmax(scores):
    "把一组得分转成概率，仅用于显示。"
    shifted = np.array(scores, dtype="float64") - max(scores)
    weights = np.exp(shifted)
    return weights / weights.sum()


if not tops:
    print("没有解析到 [TOP] 行，请先确认 §10 的 infer 已运行成功。")
else:
    count = len(images)
    fig, axes = plt.subplots(count, 2, figsize=(10.4, 3.5 * count), dpi=120)
    axes = np.atleast_2d(axes)
    for seat, row in enumerate(images):
        index = int(row["image"])
        ranked = sorted(
            (r for r in tops if int(r["image"]) == index), key=lambda r: r["rank"]
        )
        ids = [int(r["index"]) for r in ranked]
        probs = softmax([r["score"] for r in ranked])
        ref_top1 = REFERENCE[index][0][0] if index < len(REFERENCE) else -1

        left = axes[seat][0]
        with Image.open(IMAGES[index] + ".jpg") as photo:
            left.imshow(photo.convert("RGB"))
        left.axis("off")
        left.set_title(
            "%s\nNPU top-1: %s (%.1f%%)"
            % (os.path.basename(IMAGES[index]) + ".jpg", label(ids[0]), probs[0] * 100),
            fontsize=9,
        )

        right = axes[seat][1]
        seats = np.arange(len(ids))
        colors = ["tab:blue" if i == ref_top1 else "lightsteelblue" for i in ids]
        right.barh(seats, probs, color=colors)
        right.set_yticks(seats)
        right.set_yticklabels(
            ["%d %s" % (i, label(i)[:24]) for i in ids], fontsize=8
        )
        right.invert_yaxis()
        right.set_xlim(0, 1.0)
        right.set_xlabel("probability", fontsize=9)
        right.grid(True, axis="x", alpha=0.3)
        agree = ids[0] == ref_top1
        right.set_title(
            "top-5 | host reference top-1 = %d  %s"
            % (ref_top1, "MATCH" if agree else "MISMATCH"),
            fontsize=9,
            color="tab:green" if agree else "tab:red",
        )
        for seat_no, value in enumerate(probs):
            right.text(
                min(value + 0.02, 0.97), seat_no, "%.3f" % value, va="center", fontsize=8
            )
    plt.tight_layout()
    plt.show()


## 13. 一次推理的五段时间

五个数字的量级相差很远，因此要分开报出。**最后一列给出每一段相对稳态推理的倍数**，这个比值不随机器变化，比绝对值更便于比较。

表中用的是第一张图的数据；第二张图不含加载，用来对照首次推理这一项。


In [ ]:
if not stage or not loadtime:
    print("没有解析到 [STAGE] 行，请先确认 §10 的 infer 已运行成功。")
else:
    row = stage[0]
    load_ms = loadtime[0]["load_ms"]
    steady = row["steady_ms"]
    items = [
        ("模型加载（含建 Dataset）", load_ms, "只做一次"),
        ("首次推理", row["first_ms"], "含首次执行的一次性开销"),
        ("稳态推理", steady, "重复 %d 次取平均" % int(row["repeat"])),
        ("H2D 传输", row["h2d_ms"], "%d 字节" % int(row["in_bytes"])),
        ("D2H 传输", row["d2h_ms"], "%d 字节" % int(row["out_bytes"])),
    ]
    print("%-26s %12s %10s  %s" % ("阶段", "耗时 (ms)", "相对稳态", "说明"))
    print("-" * 78)
    for name, value, note in items:
        print(
            "%-27s %12.4f %9.1fx  %s" % (name, value, value / max(steady, 1e-9), note)
        )
    print()
    print(
        "端到端（H2D + 稳态推理 + D2H）：%.4f ms，%.1f FPS"
        % (row["e2e_ms"], row["fps"])
    )
    print(
        "首次推理比稳态慢 %.1f 倍；加载是稳态的 %.0f 倍。"
        % (row["first_ms"] / max(steady, 1e-9), load_ms / max(steady, 1e-9))
    )
    transfer = row["h2d_ms"] + row["d2h_ms"]
    print(
        "两个方向的传输合计 %.4f ms，占端到端的 %.1f%%；"
        "输入的字节数是输出的 %.0f 倍。"
        % (
            transfer,
            transfer / max(row["e2e_ms"], 1e-9) * 100,
            row["in_bytes"] / max(row["out_bytes"], 1),
        )
    )
    if len(stage) > 1:
        second = stage[1]
        print()
        print(
            "第一张图的首次推理 %.4f ms，第二张图的首次推理 %.4f ms。"
            "模型在两张图之间只加载了一次，两者相差 %.4f ms。"
            % (
                row["first_ms"],
                second["first_ms"],
                row["first_ms"] - second["first_ms"],
            )
        )


## 14. 运行内存的两种管理方式

同一个模型加载两次，一次由系统管理运行内存、一次由调用者管理；动态 Batch 的模型再各做一次。**§3.1 那条界线在这里由实测给出**。

`work_bytes` 与 `weight_bytes` 是模型实际占用的两块内存：权值内存由参数量决定，工作内存由中间特征图的峰值决定。


In [ ]:
def show_loads(label, rows):
    print("=" * 66)
    print(label)
    print("=" * 66)
    if not rows:
        print("（没有数据）")
        return
    for row in rows:
        owner = "系统管理" if row["owner"] == "system" else "调用者管理"
        if "load_ms" not in row:
            print(
                "  %-10s aclmdlQuerySize 返回 %d —— 查不出大小，只能由系统管理内存"
                % (owner, int(row.get("query_status", -1)))
            )
            continue
        extra = ""
        if "work_bytes" in row:
            extra = "工作内存 %.1f MB，权值内存 %.1f MB" % (
                row["work_bytes"] / 1048576,
                row["weight_bytes"] / 1048576,
            )
        print("  %-10s 加载耗时 %8.2f ms   %s" % (owner, row["load_ms"], extra))


show_loads("固定 Batch 模型", loads)
print()
show_loads("动态 Batch 模型", loads_dyn)

blocked = [r for r in loads_dyn if r["owner"] == "user" and "load_ms" not in r]
print()
if not loads_dyn:
    print("动态 Batch 模型没有数据，请先确认 §8 的转换成功。")
elif blocked:
    print("动态 Batch 模型查不出大小，只能由系统管理内存。")
else:
    print(
        "用 --dynamic_batch_size 构建的模型，可选的批大小在构建时已经逐个列举，最大批大小需要多少内存可以算出，"
        "因此 aclmdlQuerySize 查得出、调用者也可以自己管理内存，与 §3.1 一致。"
    )
    both = [r for r in loads_dyn if "work_bytes" in r] + [
        r for r in loads if "work_bytes" in r
    ]
    if len(both) == 2:
        print(
            "动态模型的工作内存是固定模型的 %.2f 倍，权值内存是 %.2f 倍 —— "
            "权重与批大小无关，工作内存随最大的批大小放大。"
            % (
                both[0]["work_bytes"] / max(both[1]["work_bytes"], 1),
                both[0]["weight_bytes"] / max(both[1]["weight_bytes"], 1),
            )
        )


## 15. 动态 Batch：时延与吞吐

四个批大小各运行一次。三列要一起看：

- `e2e_ms` 是**一次执行**的耗时，批大小越大它越长；
- `per_sample_ms` 是**每个样本**分摊到的时间，批大小越大它通常越短；
- `fps` 是吞吐，等于 $\text{batch} / \text{e2e}$。

三者不是互相独立的：**吞吐是批大小除以一次执行的时延**，而每样本耗时是吞吐的倒数。因此只要**批大小增大的倍数超过一次执行时延增大的倍数**，吞吐就会上升，同时每样本耗时下降。

**吞吐上升与单次执行时延变长同时发生**，这就是批处理的取舍：用单次执行更慢，换单位时间处理得更多。


In [ ]:
if not batch:
    print("没有解析到 [BATCH] 行，请先确认动态 Batch 模型转换成功。")
else:
    base = batch[0]
    print(
        "%6s %14s %14s %14s %12s %12s"
        % (
            "批大小",
            "一次执行 (ms)",
            "每样本 (ms)",
            "吞吐 (样本/s)",
            "相对批大小1",
            "实际输出维度",
        )
    )
    print("-" * 80)
    for row in batch:
        if "e2e_ms" not in row:
            print(
                "%6d %14s  设置失败，返回码 %d"
                % (int(row["batch"]), "-", int(row["status"]))
            )
            continue
        print(
            "%6d %14.4f %14.4f %14.1f %11.2fx %12s"
            % (
                int(row["batch"]),
                row["e2e_ms"],
                row["per_sample_ms"],
                row["fps"],
                row["fps"] / max(base["fps"], 1e-9),
                int(row["cur_dim0"]) if row.get("cur_dim0", -1) >= 0 else "未取到",
            )
        )
    ok = [r for r in batch if "e2e_ms" in r]
    if len(ok) >= 2:
        first, last = ok[0], ok[-1]
        print()
        print(
            "批大小由 %d 增到 %d：吞吐 %.2f 倍，单次执行时延 %.2f 倍，每样本耗时 %.2f 倍。"
            % (
                int(first["batch"]),
                int(last["batch"]),
                last["fps"] / max(first["fps"], 1e-9),
                last["e2e_ms"] / max(first["e2e_ms"], 1e-9),
                last["per_sample_ms"] / max(first["per_sample_ms"], 1e-9),
            )
        )
        print(
            "缓冲区按最大的批大小分配，因此 out_bytes 各个批大小相同（%d 字节）；"
            "实际用了多少要看最后一列。" % int(ok[0]["out_bytes"])
        )


批内的每个样本各自独立计算。两张图循环填满每一个批大小，因此每一个批大小的 Top-1 序列应当是这两张图的参考类别按顺序重复。**这一项要判断的是批内样本是否互不影响，以及输出中第 $i$ 段是否确实对应第 $i$ 张图。**

下面先打印输入回读校验的结果，再逐个批大小核对类别；若有不一致，会一并报出不一致的位置序号。**两项合起来可以区分差异来自数据写入还是来自设备侧的计算。**


In [ ]:
REF_TOP1 = [ranked[0][0] for ranked in REFERENCE]

for row in fills:
    print(
        "批大小 %d：输入回读校验 %s%s"
        % (
            int(row["batch"]),
            "全部位置与文件一致" if row.get("matched") == "all" else "不一致",
            ""
            if row.get("matched") == "all"
            else "，首个不一致的位置是 %d" % int(row.get("first_bad", -1)),
        )
    )
print()

if not batch_tops:
    print("没有解析到批内的 [TOP] 行。")
else:
    all_ok = True
    for row in batch:
        if "e2e_ms" not in row:
            continue
        size = int(row["batch"])
        got = [
            int(r["index"])
            for r in sorted(
                (t for t in batch_tops if int(t["batch"]) == size),
                key=lambda t: t["image"],
            )
        ]
        want = [REF_TOP1[i % len(REF_TOP1)] for i in range(size)]
        odd = [i for i, (a, b) in enumerate(zip(got, want)) if a != b]
        ok = not odd and len(got) == size
        all_ok = all_ok and ok
        print(
            "批大小 %d：Top-1 序列 %s，主机参考 %s —— %s%s"
            % (
                size,
                got,
                want,
                "PASS" if ok else "FAIL",
                "" if ok else "，不一致的位置：%s" % odd,
            )
        )
    print()
    print("批内各个批大小的类别核对：%s" % ("全部一致" if all_ok else "存在不一致"))


左右两张图用两种坐标看同一组数据。

**左图**的三条线都以批大小 1 为基准画成倍数，因此可以直接比较增长的快慢。**吞吐与单次执行的时延都在上升**：批大小增大 $N$ 倍时，一次执行的时延增加的倍数远小于 $N$，于是吞吐上升，而单次执行的时延同时变长。三条线满足一个可以在图上核对的关系：

$$\text{吞吐的倍数} = \frac{\text{批大小的倍数}}{\text{单次执行时延的倍数}}$$

吞吐的曲线之所以低于批大小那条虚线，差的正是时延增长的那部分。真正随批大小下降的是每样本耗时，它与吞吐互为倒数，画出来只是同一条信息的另一种形式，因此没有单独画。

**右图**换一组坐标：横轴是一次执行的时延，纵轴是吞吐，每个点对应一个批大小。这张图回答的是另一个问题：**给定一个时延上限，最多能拿到多少吞吐**。


In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

ok = [r for r in batch if "e2e_ms" in r]
if len(ok) < 2:
    print("可用的批大小少于两个，画不出权衡曲线。")
else:
    sizes = [int(r["batch"]) for r in ok]
    fps = [r["fps"] for r in ok]
    e2e = [r["e2e_ms"] for r in ok]

    fig, (left, right_panel) = plt.subplots(1, 2, figsize=(11.4, 4.4), dpi=120)

    # 左图：三条线都用「相对批大小 1 的倍数」表示，避免两个纵轴各自缩放造成误读。
    # 三者满足 吞吐倍数 = 批大小倍数 / 时延倍数，图上可以直接核对。
    left.plot(
        sizes, [size / sizes[0] for size in sizes], ":", color="0.55",
        label="batch size",
    )
    left.plot(
        sizes, [value / fps[0] for value in fps], "o-", color="tab:red",
        label="throughput",
    )
    left.plot(
        sizes, [value / e2e[0] for value in e2e], "s--", color="tab:blue",
        label="latency per execution",
    )
    left.axhline(1.0, color="0.85", linewidth=0.8)
    left.set_xlabel("batch size")
    left.set_ylabel("relative to batch size 1")
    left.set_xscale("log", base=2)
    left.set_xticks(sizes)
    left.set_xticklabels([str(size) for size in sizes])
    left.set_ylim(0, sizes[-1] / sizes[0] * 1.15)
    left.grid(True, alpha=0.3)
    left.legend(loc="upper left", fontsize=8)
    left.set_title("throughput = batch size / latency", fontsize=10)

    # 右图：把时延与吞吐放在同一组坐标里，一个点是一个批大小
    right_panel.plot(e2e, fps, "o-", color="tab:purple")
    for size, x, y in zip(sizes, e2e, fps):
        right_panel.annotate(
            "batch %d" % size,
            (x, y),
            textcoords="offset points",
            xytext=(7, -11),
            fontsize=8,
        )
    right_panel.set_xlabel("latency per execution (ms)")
    right_panel.set_ylabel("throughput (images/s)")
    right_panel.set_xlim(0, max(e2e) * 1.3)
    right_panel.set_ylim(0, max(fps) * 1.2)
    right_panel.grid(True, alpha=0.3)
    right_panel.set_title(
        "throughput vs latency, one point per batch size", fontsize=10
    )

    plt.tight_layout()
    plt.show()


## 16. 结果分析

### 🎓 结论

**① 模型必须先编译才能执行，编译做的三件事都只能在编译期完成。** 图优化要改变图的结构，调度优化需要对整张图的全局视野，权重重排是一次性投入换取每次推理的节省。离线模型这一名称指的就是这三件事发生在运行之前。

**② 一次推理的时间要分几段报出，因为各段的量级相差很远。** 加载只做一次；首次推理比稳态推理慢，因为第一次执行要完成一些推迟到那时才做的准备，例如算子二进制的注册与加载、设备内存的首次访问；稳态推理才是重复执行时的实际代价；两个方向的传输随数据量变化。**处理首次开销的办法是预热，并把它单独报出，而不是混入平均值**；用一个单独的数字概括推理耗时会掩盖这些差别。

**③ 传输的两个方向不对称。** 输入是一张图像的全部像素，输出只是若干个类别的得分，前者的字节数比后者大两个数量级。分类模型的输入远大于输出，因此优化传输应当优先考虑 H2D 这一侧。

**④ 运行内存的管理方式受一条约束限制，但受限的范围比字面上窄。** 约束是 Shape 不确定的模型查不出内存大小。§14 实测：**用 `--dynamic_batch_size` 构建的模型仍然查得出**，因为可选的批大小数量有限、最大批大小所需的内存可以算出。真正查不出的是用 Shape 范围构建的那一类。判断依据是编译期能不能定出内存上界，而不是第一维是否写了 `-1`。

**⑤ 释放次序是接口的要求，不取决于编码风格。** 先取地址、再释放内存、再销毁 DataBuffer：设备内存的地址只保存在 DataBuffer 中。先 Unload、再释放运行内存：否则运行时仍持有已释放的内存。两条遵循同一条原则，即销毁一个持有引用的对象之前先处理完它引用的资源。

**⑥ 可选批大小的多少是编译期开销与运行期灵活性之间的取舍。** 可选的批大小越多，`.om` 越大、编译耗时越长，运行期可选的范围也越大。运行期不能指定任意 Batch，因为 GE 要为每一个批大小编译一份执行方案。

**⑦ 增大 Batch 时，吞吐上升与单次执行时延变长同时发生。** 两者在图上都是上升的曲线，**这不是矛盾**：吞吐等于批大小除以一次执行的时延，只要批大小增大的倍数超过时延增大的倍数，吞吐就会上升。取舍体现在，单次执行变慢的代价换来了单位时间内处理更多样本。其中可以确定的一层原因是：一批样本共享一次下发与一次同步，这部分固定开销由批内各样本分摊，因此每样本耗时下降。§15 对每一个批大小逐位置核对 Top-1，用来判断**批内样本是否互不影响**，以及输出中第 $i$ 段是否确实对应第 $i$ 张图。

## 17. 🔧 动手练习

**1. 把预处理做错一步。** 分别做两次：① 去掉 §7.2 的标准化那一步；② 去掉转置那一行，直接把 HWC 的数据写出去。每次都重新运行 `infer` 与 §12。请回答：程序报错了吗？NPU 的 Top-1 与主机侧参考还一致吗？两种错法造成的偏差有什么不同？由此说明**为什么不报错不能作为预处理正确的证据**，以及应当用什么作为证据。

**2. 换一张自己的图片。** 找一张其他动物或物品的照片，用 §7.2 的同一段代码处理，跑一次 `infer` 并看 §12 的图。请回答：① Top-1 的概率与 Top-2 相差多少？② 与两张狗的照片相比，这个差值说明了什么？③ 若图片里有两个物体，模型的输出会怎样体现？

**3. 判断首次推理的开销按什么计。** §13 测出首次推理比稳态推理慢，且第二张图不再有这一项开销。设计一个实验，判断这项开销是**每个进程一次**还是**每个模型一次**：在同一个进程里加载同一个模型两遍（先 Unload 再 Load），比较第二次的首次推理。**两种结果都要能解释。**

**4. 枚举之外的批大小。** 在 `batch` 模式的批大小列表里加一个 3（构建时没有声明的值），记录 `aclmdlSetDynamicBatchSize` 的返回码，再对照 §11 中 `aclmdlGetDynamicBatch` 报出的批大小列表。**请解释为什么运行期不能任意指定 Batch。**


## 18. 🤔 思考题

**1.** ATC 做的三类优化里，权重数据重排是唯一一个不改变计算内容、只改变数据摆放的。它为什么必须在编译期做？如果放在加载期做，会付出什么代价、又能换来什么好处？

**2.** §3.1 说 Shape 不确定的模型不能由调用者管理内存，而用 `--dynamic_batch_size` 构建的动态 Batch 模型可以。**请给出一条判断准则**：拿到一个模型，怎样在不实际调用 `aclmdlQuerySize` 的情况下判断它属于哪一类？再反过来考虑：既然系统能为 Shape 不确定的模型管好内存，对固定 Shape 的模型，让调用者管理内存的意义是什么？

**3.** §16 ⑦ 陈述的是实测到的现象：批大小增大时吞吐上升、单次执行的时延变长。**请给出至少两条可能的原因**，并说明各自需要什么样的测量才能验证。再设想一个只有一个算子、且该算子完全并行的模型，这些原因是否仍然成立；由此说明在什么条件下增大 Batch 不会使单次执行的时延变长。

**4.** §16 ③ 指出输入的字节数远大于输出。若把预处理搬到设备上，H2D 传输的就不再是 602112 字节的 float32，而是几十 KB 的 JPEG。**请估算这样能减少多少传输时间**，并说明减少的这段时间在什么条件下才能变成吞吐的提升。


## 19. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 要点 | 内容 |
| --- | --- |
| 离线模型 | GE 把框架模型编译成 <code>.om</code>，做图优化、算子调度优化、权重数据重排 |
| ATC | <code>--framework</code> 选框架（5=ONNX），<code>--soc_version</code> 必须与设备一致，<code>--input_format</code> 决定数据排布；转换失败提示算子不支持时回第六章 |
| 预处理 | 等比缩放、中心裁剪、除以 255、按训练配置标准化、HWC 转 NCHW；标准化与转置做错都不报错，字节数也不变，只给出错误的类别 |
| 判据 | 主机侧用同一个 ONNX 算出参考 Top-1，与 NPU 的 Top-1 核对；正文中不写死类别标识 |
| 四种加载方式 | 文件/内存 × 系统管理/调用者管理 |
| <code>aclmdlQuerySize</code> | 出参顺序（工作内存, 权值内存）；查不出大小的是内存上界不确定的模型，用 <code>--dynamic_batch_size</code> 构建的模型不在此列 |
| 三个数据类型 | <code>aclmdlDesc</code> 规格、<code>aclmdlDataset</code> 集合、<code>aclDataBuffer</code> 单个张量；取长度用 <code>aclGetDataBufferSizeV2</code> |
| 按名字对齐 | 多输入多输出不要假定顺序，用 <code>aclmdlGetInputIndexByName</code> |
| 释放次序 | 取地址 → free → 销毁 buffer → 销毁 dataset；Unload → 销毁 desc → 释放运行内存 |
| 五段时间 | 加载、首次、稳态、H2D、D2H，量级相差很远，必须分开报出 |
| 动态 Batch | 可选的批大小在构建时确定，可用 <code>aclmdlGetDynamicBatch</code> 查出；选择器输入名固定为 <code>ACL_DYNAMIC_TENSOR_NAME</code>，其内存不要写入 |
| 缓冲区与实际用量 | 按最大的批大小分配，用 <code>aclmdlGetCurOutputDims</code> 取本次的实际维度 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">离线模型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">GE 把框架模型编译成 <code>.om</code>，做图优化、算子调度优化、权重数据重排</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">ATC</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>--framework</code> 选框架（5=ONNX），<code>--soc_version</code> 必须与设备一致，<code>--input_format</code> 决定数据排布；转换失败提示算子不支持时回第六章</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">预处理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">等比缩放、中心裁剪、除以 255、按训练配置标准化、HWC 转 NCHW；标准化与转置做错都不报错，字节数也不变，只给出错误的类别</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">判据</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机侧用同一个 ONNX 算出参考 Top-1，与 NPU 的 Top-1 核对；正文中不写死类别标识</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">四种加载方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">文件/内存 × 系统管理/调用者管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlQuerySize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">出参顺序（工作内存, 权值内存）；查不出大小的是内存上界不确定的模型，用 <code>--dynamic_batch_size</code> 构建的模型不在此列</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">三个数据类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclmdlDesc</code> 规格、<code>aclmdlDataset</code> 集合、<code>aclDataBuffer</code> 单个张量；取长度用 <code>aclGetDataBufferSizeV2</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按名字对齐</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多输入多输出不要假定顺序，用 <code>aclmdlGetInputIndexByName</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">释放次序</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">取地址 → free → 销毁 buffer → 销毁 dataset；Unload → 销毁 desc → 释放运行内存</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">五段时间</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">加载、首次、稳态、H2D、D2H，量级相差很远，必须分开报出</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">动态 Batch</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可选的批大小在构建时确定，可用 <code>aclmdlGetDynamicBatch</code> 查出；选择器输入名固定为 <code>ACL_DYNAMIC_TENSOR_NAME</code>，其内存不要写入</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">缓冲区与实际用量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按最大的批大小分配，用 <code>aclmdlGetCurOutputDims</code> 取本次的实际维度</td>
</tr>
</tbody>
</table>

§1.2 提出的三个问题，本实验都给出了实测的回答：时间分布在加载、首次、稳态与两个方向的传输上，各段量级不同；运行内存由谁管理，取决于编译期能否定出内存上界，而用 `--dynamic_batch_size` 构建的动态 Batch 模型并不受这条约束限制；增大批大小时吞吐与单次执行的时延同时上升，而前者上升得更快。
